In [ ]:
# ABLATION STUDY 6 CẤU HÌNH
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score, accuracy_score
from pathlib import Path
import timm

In [ ]:
# CẤU HÌNH
 
RESNET_PATH = "/kaggle/input/notebooks/dunnguynvn/train-resnet50-ver2/resnet50_output/resnet50_best.pth"
SWIN_PATH   = "/kaggle/input/notebooks/dunnguynvn/trainswinver2/swin_output/swin_best.pth"
SPLIT_PATH  = "/kaggle/input/datasets/dunnguynvn/blood-data-l2/dataset_split"
OUTPUT_DIR  = "/kaggle/working/ablation_output"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
 
CLASSES     = ["BA","BNE","EO","ERB","IG","LY","MMY",
               "MO","MY","MYELOBLAST","PLATELET","PMY","SNE"]
HARD_CLASSES= ["MY", "MMY", "PMY"]
NUM_CLASSES = len(CLASSES)
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# LOAD MÔ HÌNH

def build_resnet50(n):
    m = models.resnet50(weights=None)
    m.fc = nn.Sequential(
        nn.Dropout(0.3), nn.Linear(m.fc.in_features, 512),
        nn.ReLU(inplace=True), nn.Dropout(0.2), nn.Linear(512, n)
    )
    return m

def build_swin(n):
    bb = timm.create_model("swin_small_patch4_window7_224",
                           pretrained=False, num_classes=0, global_pool="avg")
    class S(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = bb
            self.head = nn.Sequential(
                nn.Dropout(0.3), nn.Linear(bb.num_features, 384),
                nn.ReLU(inplace=True), nn.Dropout(0.2), nn.Linear(384, n)
            )
        def forward(self, x): return self.head(self.backbone(x))
    return S()

resnet_model = build_resnet50(NUM_CLASSES).to(DEVICE)
swin_model   = build_swin(NUM_CLASSES).to(DEVICE)

ckpt_r = torch.load(RESNET_PATH, map_location=DEVICE)
ckpt_s = torch.load(SWIN_PATH,   map_location=DEVICE)
resnet_model.load_state_dict(ckpt_r["model_state"])
swin_model.load_state_dict(ckpt_s["model_state"])
resnet_model.eval(); swin_model.eval()


In [ ]:
# DATALOADER

tfm = transforms.Compose([
    transforms.Resize((224, 224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

test_ds = datasets.ImageFolder(os.path.join(SPLIT_PATH,"test"), transform=tfm)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)


In [ ]:
# INFERENCE

all_lr, all_ls, all_labels = [], [], []

with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(DEVICE)
        all_lr.append(resnet_model(imgs).cpu().numpy())
        all_ls.append(swin_model(imgs).cpu().numpy())
        all_labels.extend(lbls.numpy())

logits_r    = np.vstack(all_lr)
logits_s    = np.vstack(all_ls)
true_labels = np.array(all_labels)

In [ ]:
# UTIL

def softmax(x, T=1.0):
    x = x / T
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def evaluate(probs, labels, name):
    preds  = probs.argmax(axis=1)
    acc    = accuracy_score(labels, preds)
    f1_mac = f1_score(labels, preds, average="macro", zero_division=0)

    hard_f1s = [
        f1_score(labels==CLASSES.index(c), preds==CLASSES.index(c), zero_division=0)
        for c in HARD_CLASSES
    ]
    hard_avg = np.mean(hard_f1s)

    print(f"{name:<45} Acc={acc*100:.2f}%  F1={f1_mac:.4f}  Hard={hard_avg:.4f}")

    return {
        "config": name,
        "accuracy_pct": round(acc*100,2),
        "f1_macro": round(f1_mac,4),
        "f1_hard_avg": round(hard_avg,4)
    }

In [ ]:
# ABLATION

results = []

# C1
p_r1 = softmax(logits_r)
p_s1 = softmax(logits_s)
results.append(evaluate(0.5*p_r1 + 0.5*p_s1, true_labels,
    "C1: Baseline (0.5/0.5)"))

# C2 (grid search result)
p_r2 = softmax(logits_r)
p_s2 = softmax(logits_s)
results.append(evaluate(0.7*p_r2 + 0.3*p_s2, true_labels,
    "C2: Grid Search weight (0.7/0.3 from val)"))

# C3
p_r3 = softmax(logits_r, T=1.2)
p_s3 = softmax(logits_s)
results.append(evaluate(0.7*p_r3 + 0.3*p_s3, true_labels,
    "C3: + Temperature Scaling"))

# C4 (FINAL MODEL)
p_r4 = softmax(logits_r, T=1.2)
p_s4 = softmax(logits_s)
p_c4 = 0.7*p_r4 + 0.3*p_s4

for cls in HARD_CLASSES:
    idx = CLASSES.index(cls)
    p_c4[:, idx] = 0.3*p_r4[:, idx] + 0.7*p_s4[:, idx]

results.append(evaluate(p_c4, true_labels,
    "C4: Adaptive Weighting (FINAL MODEL)"))

# C5 (reference)
p_r5 = softmax(logits_r, T=1.2)
p_s5 = softmax(logits_s)

conf_r = p_r5.max(axis=1, keepdims=True)
conf_s = p_s5.max(axis=1, keepdims=True)

w_r = conf_r / (conf_r + conf_s)
w_s = conf_s / (conf_r + conf_s)

p_c5 = w_r * p_r5 + w_s * p_s5

for cls in HARD_CLASSES:
    idx = CLASSES.index(cls)
    p_c5[:, idx] = 0.3*p_r5[:, idx] + 0.7*p_s5[:, idx]

results.append(evaluate(p_c5, true_labels,
    "C5: Confidence-based (reference)"))

In [ ]:
#IN KẾT QUẢ

best_f1 = max(x["f1_macro"] for x in results)

print("\nKẾT QUẢ")
for r in results:
    if r["config"].startswith("C4"):
        marker = " <-- BEST (FINAL)"
    elif r["f1_macro"] == best_f1 and not any(
        x["config"].startswith("C4") and x["f1_macro"] == best_f1 for x in results
    ):
        marker = " <-- BEST"
    else:
        marker = ""

    print(f"{r['config']:<45} {r['accuracy_pct']:>6.2f}%  "
          f"{r['f1_macro']:.4f}  {r['f1_hard_avg']:.4f}{marker}")